# OpenCoesione 2014-2020 - preanalysis v0

Prima lettura su `fondi UE 2014-2020` per `regione x tema`.

Domanda chiave:
- Per regione e per tema, quanta parte dei fondi UE del ciclo 2014-2020 e stata effettivamente pagata?

Cautela:
- `ratio_spesa > 1` non va letto subito come errore; puo indicare pagamenti che superano la quota UE osservata nel taglio scelto e richiede verifica metodologica.


In [1]:
from pathlib import Path

import duckdb
import pandas as pd

pd.options.display.float_format = '{:,.3f}'.format

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'preprojects').exists() and (candidate / 'registry').exists():
            return candidate
    raise RuntimeError('Repo root non trovato')

REPO_ROOT = find_repo_root(Path.cwd())
MART_PATH = REPO_ROOT / 'out' / 'data' / 'mart' / 'opencoesione_pagamenti_ue_2014_2020' / '2020' / 'mart_regione_tema.parquet'
con = duckdb.connect()
MART_PATH


WindowsPath('C:/Users/gabry/OneDrive/Desktop/dataciviclab-workspace/dataset-incubator/out/data/mart/opencoesione_pagamenti_ue_2014_2020/2020/mart_regione_tema.parquet')

## 1. Copertura del mart

Verifichiamo che il mart sia davvero ristretto a regioni italiane e temi sintetici.

In [2]:
coverage = con.execute(
    """
    select
        count(*) as righe,
        count(distinct regione) as regioni,
        count(distinct tema) as temi,
        sum(case when regione like '%:::%' then 1 else 0 end) as multi_regione,
        sum(case when regione in ('AMBITO NAZIONALE', 'PAESI EUROPEI') then 1 else 0 end) as non_regione
    from read_parquet($path)
    """,
    {'path': str(MART_PATH)},
).fetchdf()
coverage


,righe,regioni,temi,multi_regione,non_regione
0,220,20,11,0.000,0.000


## 2. Regioni con ratio di spesa piu basso

Aggregazione ponderata sul totale UE per regione.

In [3]:
regioni_ratio = con.execute(
    """
    select
        regione,
        sum(finanz_ue_tot) as finanz_ue_tot,
        sum(tot_pagamenti_tot) as tot_pagamenti_tot,
        sum(tot_pagamenti_tot) / sum(finanz_ue_tot) as ratio_spesa
    from read_parquet($path)
    group by 1
    order by ratio_spesa asc, finanz_ue_tot desc
    """,
    {'path': str(MART_PATH)},
).fetchdf()
regioni_ratio


,regione,finanz_ue_tot,tot_pagamenti_tot,ratio_spesa
0,SICILIA,"6,774,941,920.940","6,144,567,157.450",0.907
1,PUGLIA,"7,542,183,615.400","7,138,601,303.730",0.946
2,CALABRIA,"2,924,955,140.430","2,845,228,676.370",0.973
3,MOLISE,"150,903,357.800","157,145,843.230",1.041
4,CAMPANIA,"8,144,627,808.870","8,850,063,339.070",1.087
5,BASILICATA,"1,061,861,352.120","1,217,464,925.860",1.147
6,TOSCANA,"1,746,289,988.580","2,239,203,491.030",1.282
7,ABRUZZO,"506,848,042.500","683,150,379.690",1.348
8,LAZIO,"1,809,354,786.000","2,457,896,604.550",1.358
9,VENETO,"1,162,042,274.690","1,610,389,392.110",1.386


## 3. Temi con ratio di spesa piu basso

Qui guardiamo il rapporto pagato/assegnato per tema, sempre in forma ponderata.

In [4]:
temi_ratio = con.execute(
    """
    select
        tema,
        sum(finanz_ue_tot) as finanz_ue_tot,
        sum(tot_pagamenti_tot) as tot_pagamenti_tot,
        sum(tot_pagamenti_tot) / sum(finanz_ue_tot) as ratio_spesa
    from read_parquet($path)
    group by 1
    order by ratio_spesa asc, finanz_ue_tot desc
    """,
    {'path': str(MART_PATH)},
).fetchdf()
temi_ratio


,tema,finanz_ue_tot,tot_pagamenti_tot,ratio_spesa
0,Ambiente,"3,726,336,666.590","3,432,584,815.130",0.921
1,Trasporti e mobilità,"4,898,241,403.300","4,878,213,925.810",0.996
2,Energia,"1,959,581,296.550","2,063,714,204.390",1.053
3,Cultura e turismo,"1,111,716,517.770","1,237,954,962.920",1.114
4,Reti e servizi digitali,"3,250,695,011.560","3,646,783,022.560",1.122
5,Competitività delle imprese,"6,190,665,581.350","7,170,677,228.220",1.158
6,Capacità amministrativa,"1,579,039,729.540","1,831,368,899.230",1.160
7,Occupazione e lavoro,"4,187,285,908.670","5,176,845,622.070",1.236
8,Istruzione e formazione,"4,890,523,775.150","6,082,090,066.480",1.244
9,Inclusione sociale e salute,"4,662,045,092.020","5,878,393,003.030",1.261


## 4. Casi specifici con ratio basso ma base non trascurabile

Escludiamo i casi microscopici e teniamo solo combinazioni con almeno 1 milione di euro di finanziamento UE.

In [5]:
casi_bassi = con.execute(
    """
    select *
    from read_parquet($path)
    where finanz_ue_tot >= 1000000
    order by ratio_spesa asc, finanz_ue_tot desc
    limit 15
    """,
    {'path': str(MART_PATH)},
).fetchdf()
casi_bassi


,regione,tema,n_progetti,finanz_ue_tot,tot_pagamenti_tot,ratio_spesa
0,EMILIA-ROMAGNA,Ambiente,20,"3,283,774.920","1,693,911.360",0.516
1,TRENTINO-ALTO ADIGE,Cultura e turismo,6,"3,365,774.800","1,762,767.000",0.524
2,SICILIA,Cultura e turismo,167,"154,812,763.650","86,935,399.290",0.562
3,BASILICATA,Ambiente,117,"79,809,764.730","49,974,982.410",0.626
4,LAZIO,Ambiente,91,"49,840,433.020","33,073,362.840",0.664
5,SICILIA,Ambiente,355,"610,703,999.730","405,445,355.840",0.664
6,CALABRIA,Ambiente,480,"397,277,542.620","279,930,410.340",0.705
7,CAMPANIA,Energia,535,"340,239,145.320","243,295,043.070",0.715
8,MOLISE,Energia,71,"13,333,210.920","9,844,888.010",0.738
9,CALABRIA,Inclusione sociale e salute,531,"194,777,369.560","145,720,991.190",0.748


## 5. Lettura v0

Prima lettura prudente:

- il mart e gia abbastanza pulito da sostenere una prima domanda pubblica
- `Ambiente` emerge come uno dei temi con ratio piu basso a livello aggregato
- alcune regioni del Mezzogiorno, soprattutto `Sicilia`, `Puglia` e `Calabria`, stanno relativamente piu in basso nella classifica dei ratio ponderati
- alcuni casi con ratio molto basso e base piccola o molto specifica vanno trattati come follow-up, non come finding pubblico principale
- i casi con `ratio_spesa > 1` richiedono una nota metodologica dedicata prima di qualsiasi lettura forte
